# Solution: Pandas ETL to Snowflake

Build a full pipeline: extract, profile, rename, clean, validate, engineer, analyze, publish, and reconcile.

This is an AI-Free Zone activity. Write and explain your own code. Never place a password, token, AWS key, or populated connection value in this notebook.

## How this notebook teaches

- **Run as written:** setup or unfamiliar mechanics are provided.
- **Complete a focused TODO:** some code is provided and you add the missing pandas operation.
- **Write the entire cell:** later work gives an exact contract and expects you to assemble the solution.

The challenge should come from applying skills, not guessing an unstated formula or connection pattern.


## 1. Environment and imports

**Run as written.** The notebook uses `ConfigParser` to read local Snowflake context from `snow.cfg`. The connection file is not imported into the notebook and is ignored by the repository-root `.gitignore`.


In [ ]:
from configparser import ConfigParser
from pathlib import Path

import numpy as np
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Extract and profile

**Run as written.** Read the approved public CSV from S3. The expected source shape is 463,152 rows and 27 columns.


In [ ]:
ACCIDENTS_S3_URI = (
    "s3://techcatalyst-de-2026/raw/accidents/"
    "accidents_2017_to_2023_english.csv"
)

raw_accidents = pd.read_csv(
    ACCIDENTS_S3_URI,
    storage_options={"anon": True},
)

print({"rows": len(raw_accidents), "columns": len(raw_accidents.columns)})
assert raw_accidents.shape == (463_152, 27), (
    "The source shape changed. Stop and confirm the approved file."
)
raw_accidents.head()


### Your work: profile the source

Write this entire cell.

1. Display the list of column names.
2. Display a two-column table containing each source column and dtype.
3. Display the first five rows.
4. Add two comments identifying fields that need type correction.

**Expected observations:** `inverse_data` is text but represents a date. `km` contains comma decimal notation and must become numeric.


In [ ]:
print(raw_accidents.columns.tolist())
display(raw_accidents.dtypes.rename("source_dtype").to_frame())
display(raw_accidents.head())

# inverse_data is text and must become datetime.
# km contains comma decimal notation and must become numeric.


## 3. Build the rename mapping

The source uses several unclear or misspelled names. Build a Python dictionary named `COLUMN_MAP` with these exact source-to-target pairs:

| Source column | Target column |
|---|---|
| `inverse_data` | `accident_date` |
| `week_day` | `day_of_week` |
| `hour` | `accident_time` |
| `state` | `state_code` |
| `road_id` | `highway_number` |
| `km` | `kilometer_marker` |
| `city` | `city_name` |
| `cause_of_accident` | `accident_cause` |
| `type_of_accident` | `accident_type` |
| `victims_condition` | `casualty_status` |
| `weather_timestamp` | `daylight_condition` |
| `road_direction` | `traffic_direction` |
| `wheather_condition` | `weather_condition` |
| `road_type` | `road_classification` |
| `road_delineation` | `road_geometry` |
| `people` | `total_people_involved` |
| `deaths` | `fatalities` |
| `slightly_injured` | `minor_injuries` |
| `severely_injured` | `serious_injuries` |
| `uninjured` | `uninjured_count` |
| `ignored` | `unknown_status` |
| `total_injured` | `total_injuries` |
| `vehicles_involved` | `vehicle_count` |
| `latitude` | `accident_latitude` |
| `longitude` | `accident_longitude` |
| `regional` | `regional_office` |
| `police_station` | `reporting_station` |

Two entries are started for you. Add the other 25. Do not copy a completed dictionary from the solution folder.


In [ ]:
COLUMN_MAP = {
    "inverse_data": "accident_date",
    "week_day": "day_of_week",
    "hour": "accident_time",
    "state": "state_code",
    "road_id": "highway_number",
    "km": "kilometer_marker",
    "city": "city_name",
    "cause_of_accident": "accident_cause",
    "type_of_accident": "accident_type",
    "victims_condition": "casualty_status",
    "weather_timestamp": "daylight_condition",
    "road_direction": "traffic_direction",
    "wheather_condition": "weather_condition",
    "road_type": "road_classification",
    "road_delineation": "road_geometry",
    "people": "total_people_involved",
    "deaths": "fatalities",
    "slightly_injured": "minor_injuries",
    "severely_injured": "serious_injuries",
    "uninjured": "uninjured_count",
    "ignored": "unknown_status",
    "total_injured": "total_injuries",
    "vehicles_involved": "vehicle_count",
    "latitude": "accident_latitude",
    "longitude": "accident_longitude",
    "regional": "regional_office",
    "police_station": "reporting_station",
}


### Apply and verify the mapping

Complete the focused TODOs:

1. Use `raw_accidents.rename(columns=COLUMN_MAP)` and copy the result into `renamed_accidents`.
2. Calculate which required columns are missing.
3. Assert that the missing set is empty.

If the assertion fails, compare your dictionary with the instruction table. Do not continue with a partially renamed schema.


In [ ]:
REQUIRED_ANALYSIS_COLUMNS = {
    "accident_date",
    "day_of_week",
    "accident_time",
    "state_code",
    "accident_cause",
    "weather_condition",
    "road_classification",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
}

renamed_accidents = raw_accidents.rename(columns=COLUMN_MAP).copy()
missing_required = REQUIRED_ANALYSIS_COLUMNS.difference(renamed_accidents.columns)
assert not missing_required, f"Missing required columns: {sorted(missing_required)}"

print(f"Mapped {len(COLUMN_MAP)} source columns.")
display(renamed_accidents.head())


### Rename checkpoint

Your mapping must contain 27 entries. `renamed_accidents` should still have 463,152 rows and 27 columns. Its first fields should be `accident_date`, `day_of_week`, `accident_time`, and `state_code`.


## 4. Clean types and values

This function is intentionally mixed:

- The date conversion is completed as an example.
- The time parser is provided, and you create `accident_hour`.
- The comma-decimal requirement and pandas methods are stated.
- The loops are provided, and you complete the conversion expressions.
- The required-row list is provided so you do not guess which missing values justify dropping a row.

### Required behavior

1. Convert `accident_date` with `pd.to_datetime(..., errors="coerce")`.
2. Parse `accident_time` with format `%H:%M:%S` and store the hour as nullable `Int64`.
3. Replace commas with periods in `kilometer_marker` before `pd.to_numeric`.
4. Trim and lowercase the listed text fields.
5. Trim and uppercase `state_code`.
6. Convert the listed measures with `pd.to_numeric(..., errors="coerce")`.
7. Drop rows only when a field required by the four analyses is unusable.


In [ ]:
TEXT_COLUMNS = [
    "day_of_week",
    "city_name",
    "accident_cause",
    "accident_type",
    "casualty_status",
    "daylight_condition",
    "traffic_direction",
    "weather_condition",
    "road_classification",
    "road_geometry",
]

NUMERIC_COLUMNS = [
    "total_people_involved",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "uninjured_count",
    "unknown_status",
    "total_injuries",
    "vehicle_count",
]

REQUIRED_AFTER_CLEANING = [
    "accident_date",
    "accident_hour",
    "day_of_week",
    "state_code",
    "accident_cause",
    "weather_condition",
    "road_classification",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
]


In [ ]:
def clean_accidents(frame):
    cleaned = frame.copy()

    # Completed example: text date to datetime.
    cleaned["accident_date"] = pd.to_datetime(
        cleaned["accident_date"],
        errors="coerce",
    )

    # The parser is provided. Extract the nullable integer hour.
    parsed_time = pd.to_datetime(
        cleaned["accident_time"].astype("string"),
        format="%H:%M:%S",
        errors="coerce",
    )
    cleaned["accident_hour"] = parsed_time.dt.hour.astype("Int64")

    cleaned["kilometer_marker"] = pd.to_numeric(
        cleaned["kilometer_marker"]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )

    for column in TEXT_COLUMNS:
        cleaned[column] = (
            cleaned[column].astype("string").str.strip().str.lower()
        )

    cleaned["state_code"] = (
        cleaned["state_code"].astype("string").str.strip().str.upper()
    )

    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    rows_before = len(cleaned)
    cleaned = cleaned.dropna(subset=REQUIRED_AFTER_CLEANING).copy()
    print({
        "rows_before": rows_before,
        "rows_after": len(cleaned),
        "rows_removed": rows_before - len(cleaned),
    })
    return cleaned

In [ ]:
cleaned_accidents = clean_accidents(renamed_accidents)

display(cleaned_accidents.head())
display(cleaned_accidents.dtypes.rename("cleaned_dtype").to_frame())

assert len(cleaned_accidents) == 463_152
assert str(cleaned_accidents["accident_hour"].dtype) == "Int64"
assert pd.api.types.is_datetime64_any_dtype(cleaned_accidents["accident_date"])


### Cleaning checkpoint

Expected evidence:

- Rows before: 463,152
- Rows after: 463,152
- Rows removed: 0
- `accident_date` is datetime
- `accident_hour` is nullable `Int64`
- `kilometer_marker` is numeric
- Text categories are trimmed and lowercase
- `state_code` is uppercase

Rows with missing `reporting_station`, `highway_number`, `kilometer_marker`, or `regional_office` remain because those fields are not required by the four analyses.


## 5. Data-quality evidence

### 5A. Missing-value profile

Write a table with `column_name`, `missing_count`, and `missing_percent`. Keep only columns with at least one missing value and sort from most missing to least.


In [ ]:
missing_profile = (
    cleaned_accidents.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(
        missing_percent=lambda frame:
            frame["missing_count"] / len(cleaned_accidents) * 100
    )
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
    .rename_axis("column_name")
    .reset_index()
)

display(missing_profile)


**Expected missing counts:** reporting station 1,310; highway number 990; kilometer marker 990; regional office 10.


### 5B. Injury-integrity check

The source field `total_injuries` represents minor plus serious injuries. It does not include fatalities.

Create:

`calculated_total_injuries = minor_injuries + serious_injuries`

Then create `injury_total_match` and report the mismatch count and percentage.


In [ ]:
cleaned_accidents["calculated_total_injuries"] = (
    cleaned_accidents["minor_injuries"]
    + cleaned_accidents["serious_injuries"]
)
cleaned_accidents["injury_total_match"] = (
    cleaned_accidents["calculated_total_injuries"]
    == cleaned_accidents["total_injuries"]
)

injury_mismatches = int((~cleaned_accidents["injury_total_match"]).sum())
mismatch_percent = injury_mismatches / len(cleaned_accidents) * 100

print({
    "injury_mismatches": injury_mismatches,
    "mismatch_percent": round(mismatch_percent, 4),
})
assert injury_mismatches == 0


### 5C. IQR outlier report

You are not expected to invent the IQR method from memory. Read the method, run the provided implementation, and interpret the evidence.

For one numerical column:

1. `Q1` is the 25th percentile.
2. `Q3` is the 75th percentile.
3. `IQR = Q3 - Q1`.
4. The lower boundary is `Q1 - 1.5 * IQR`.
5. The upper boundary is `Q3 + 1.5 * IQR`.
6. Values below the lower boundary or above the upper boundary are flagged for review.

An IQR flag is not proof that a record is wrong. For example, any fatality is statistically unusual when the 75th percentile is zero, but it may still be valid. Report these rows. Do not automatically delete them.


In [ ]:
OUTLIER_COLUMNS = [
    "total_people_involved",
    "fatalities",
    "minor_injuries",
    "serious_injuries",
    "total_injuries",
    "vehicle_count",
]


def build_iqr_report(frame, columns):
    records = []

    for column in columns:
        series = frame[column].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (
            (series < lower_bound)
            | (series > upper_bound)
        )

        records.append({
            "column_name": column,
            "q1": q1,
            "q3": q3,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_mask.sum()),
            "outlier_percent": round(outlier_mask.mean() * 100, 2),
            "max_value": series.max(),
        })

    return pd.DataFrame(records)


outlier_report = build_iqr_report(
    cleaned_accidents,
    OUTLIER_COLUMNS,
)
display(outlier_report)

### IQR checkpoint and interpretation

Expected outlier counts:

| Column | Outlier count | Maximum |
|---|---:|---:|
| total_people_involved | 8,587 | 80 |
| fatalities | 31,344 | 21 |
| minor_injuries | 20,283 | 61 |
| serious_injuries | 96,751 | 31 |
| total_injuries | 30,484 | 66 |
| vehicle_count | 8,574 | 23 |

Add a Markdown cell below and answer:

1. Why does `fatalities` have an upper IQR boundary of zero?
2. Why should that not cause every fatal accident to be deleted?
3. Which maximum value would you review first, and what additional evidence would you request?


## 6. Feature engineering

Create:

- `year`, `month`, `day`, and `quarter` from `accident_date`
- `time_of_day_category`: morning 05:00 through 11:59, afternoon 12:00 through 16:59, evening 17:00 through 20:59, night otherwise
- `weekend_flag`
- `fatal_accident_flag`
- `multi_vehicle_flag`
- `accident_severity_score = fatalities * 10 + serious_injuries * 3 + minor_injuries`

The `np.select` conditions and labels are provided. Complete the focused assignments.


In [ ]:
def engineer_features(frame):
    featured = frame.copy()

    # The year line is the example. Complete the other date parts.
    featured["year"] = featured["accident_date"].dt.year
    featured["month"] = featured["accident_date"].dt.month
    featured["day"] = featured["accident_date"].dt.day
    featured["quarter"] = featured["accident_date"].dt.quarter

    time_conditions = [
        featured["accident_hour"].between(5, 11),
        featured["accident_hour"].between(12, 16),
        featured["accident_hour"].between(17, 20),
    ]
    time_labels = ["morning", "afternoon", "evening"]
    featured["time_of_day_category"] = np.select(
        time_conditions,
        time_labels,
        default="night",
    )

    featured["weekend_flag"] = (
        featured["day_of_week"].isin(["saturday", "sunday"])
    )
    featured["fatal_accident_flag"] = featured["fatalities"] > 0
    featured["multi_vehicle_flag"] = featured["vehicle_count"] > 1
    featured["accident_severity_score"] = (
        featured["fatalities"] * 10
        + featured["serious_injuries"] * 3
        + featured["minor_injuries"]
    )

    return featured

In [ ]:
featured_accidents = engineer_features(cleaned_accidents)

FEATURE_COLUMNS = [
    "year",
    "month",
    "day",
    "quarter",
    "time_of_day_category",
    "weekend_flag",
    "fatal_accident_flag",
    "multi_vehicle_flag",
    "accident_severity_score",
]

display(featured_accidents[FEATURE_COLUMNS].head())
assert set(FEATURE_COLUMNS).issubset(featured_accidents.columns)


### Feature checkpoint

The first row should represent January 1, 2017, during the night, on a weekend. Its severity score is 4. The first five rows should have severity scores 4, 10, 14, 10, and 5.


## 7. Create four business analysis DataFrames

Each analysis specifies its grain, exact output columns, sorting rule, and expected row count. Counts remain beside averages so a rare category is not mistaken for a reliable pattern.


### Analysis 1: Severity by weather and road classification

**Grain:** one row per weather and road-classification combination.

**Steps:**

1. Group by `weather_condition` and `road_classification`.
2. Aggregate `accident_count` with size.
3. Aggregate `average_severity_score` with mean.
4. Reset the index.
5. Round the average to two decimals.
6. Sort by average severity descending, then accident count descending.

**Required columns:** `weather_condition`, `road_classification`, `accident_count`, `average_severity_score`.  
**Expected rows:** 28.


In [ ]:
severity_analysis = (
    featured_accidents
    .groupby(
        ["weather_condition", "road_classification"],
        dropna=False,
    )
    .agg(
        accident_count=("accident_date", "size"),
        average_severity_score=("accident_severity_score", "mean"),
    )
    .reset_index()
    .assign(
        average_severity_score=lambda frame:
            frame["average_severity_score"].round(2)
    )
    .sort_values(
        ["average_severity_score", "accident_count"],
        ascending=[False, False],
    )
)

display(severity_analysis.head())
assert len(severity_analysis) == 28


**Checkpoint:** the first row should be fog plus simple road classification, with 2,482 accidents and an average severity score of 3.51.


### Analysis 2: Fatal accidents by time and weekend status

**Grain:** one row per time-of-day and weekend combination.

The filter is provided. Complete the grouping, count, reset, and descending sort.

**Required columns:** `time_of_day_category`, `weekend_flag`, `number_of_fatal_accidents`.  
**Expected rows:** 8.


In [ ]:
fatal_accidents = featured_accidents.loc[
    featured_accidents["fatal_accident_flag"]
].copy()

fatal_temporal_patterns = (
    fatal_accidents
    .groupby(
        ["time_of_day_category", "weekend_flag"],
        dropna=False,
    )
    .size()
    .rename("number_of_fatal_accidents")
    .reset_index()
    .sort_values("number_of_fatal_accidents", ascending=False)
)

display(fatal_temporal_patterns)
assert len(fatal_temporal_patterns) == 8


**Checkpoint:** the largest group is weekday evening with 5,447 fatal accidents.


### Analysis 3: Accident cause and vehicle involvement

**Write the entire cell.**

**Grain:** one row per accident cause and multi-vehicle flag.

**Required columns:** `accident_cause`, `multi_vehicle_flag`, `number_of_accidents`, `average_severity_score`.

Use named aggregation, round the average to two decimals, and sort by accident count descending, then average severity descending.

**Expected rows:** 167.


In [ ]:
cause_vehicle_analysis = (
    featured_accidents
    .groupby(
        ["accident_cause", "multi_vehicle_flag"],
        dropna=False,
    )
    .agg(
        number_of_accidents=("accident_date", "size"),
        average_severity_score=("accident_severity_score", "mean"),
    )
    .reset_index()
    .assign(
        average_severity_score=lambda frame:
            frame["average_severity_score"].round(2)
    )
    .sort_values(
        ["number_of_accidents", "average_severity_score"],
        ascending=[False, False],
    )
)

display(cause_vehicle_analysis.head())
assert len(cause_vehicle_analysis) == 167


**Checkpoint:** the first group is multi-vehicle accidents caused by driver's lack of attention to conveyance, with 71,133 accidents and an average severity score of 2.25.


### Analysis 4: Quarterly state hotspots

**Grain:** one row per state and quarter.

**Required columns:** `state_code`, `quarter`, `total_accidents`.

Group by the two grain columns, count with `size()`, rename the count, reset the index, and sort descending.

**Expected rows:** 108.


In [ ]:
quarterly_hotspot_analysis = (
    featured_accidents
    .groupby(["state_code", "quarter"], dropna=False)
    .size()
    .rename("total_accidents")
    .reset_index()
    .sort_values("total_accidents", ascending=False)
)

display(quarterly_hotspot_analysis.head())
assert len(quarterly_hotspot_analysis) == 108


**Checkpoint:** the first row is MG, quarter 1, with 16,266 accidents.


### Validate all four outputs

The analysis registry and grain columns are provided. Build one quality row per output with:

- table name
- row count
- column count
- duplicate count at the stated grain
- null count in the grain columns

Then assert that every output is nonempty and has no duplicate grain rows.


In [ ]:
ANALYSES = {
    "SEVERITY_ANALYSIS": (
        severity_analysis,
        ["weather_condition", "road_classification"],
    ),
    "FATAL_TEMPORAL_PATTERNS": (
        fatal_temporal_patterns,
        ["time_of_day_category", "weekend_flag"],
    ),
    "CAUSE_VEHICLE_ANALYSIS": (
        cause_vehicle_analysis,
        ["accident_cause", "multi_vehicle_flag"],
    ),
    "QUARTERLY_HOTSPOT_ANALYSIS": (
        quarterly_hotspot_analysis,
        ["state_code", "quarter"],
    ),
}

analysis_quality_summary = pd.DataFrame([
    {
        "table_name": table_name,
        "row_count": len(frame),
        "column_count": len(frame.columns),
        "duplicate_grain_rows": int(frame.duplicated(grain).sum()),
        "null_grain_values": int(frame[grain].isna().sum().sum()),
    }
    for table_name, (frame, grain) in ANALYSES.items()
])

display(analysis_quality_summary)
assert (analysis_quality_summary["row_count"] > 0).all()
assert (analysis_quality_summary["duplicate_grain_rows"] == 0).all()
assert (analysis_quality_summary["null_grain_values"] == 0).all()


### Analysis checkpoint

| Table | Rows | Columns | Duplicate grain rows | Null grain values |
|---|---:|---:|---:|---:|
| SEVERITY_ANALYSIS | 28 | 4 | 0 | 0 |
| FATAL_TEMPORAL_PATTERNS | 8 | 3 | 0 | 0 |
| CAUSE_VEHICLE_ANALYSIS | 167 | 4 | 0 | 0 |
| QUARTERLY_HOTSPOT_ANALYSIS | 108 | 3 | 0 | 0 |


## 8. Load Snowflake context from `snow.cfg`

This is your first time connecting to Snowflake from Python, so read each piece before running it.

- `ConfigParser` reads the `snow.cfg` text file, so your account and password never live in this notebook.
- The `[DEV]` section becomes a plain Python dictionary named `params`.
- `snowflake.connector.connect(**params)` opens the connection. `**params` spreads the dictionary into named arguments, so it becomes `connect(account=..., user=..., password=..., ...)`.

Your `student-work/week4/day4/snow.cfg` should look like this:

```ini
[DEV]
account = <your_snowflake_account_identifier>
user = <your_snowflake_username>
password = <your_snowflake_password>
role = DE
warehouse = COMPUTE_WH
database = TECHCATALYST
schema = <your_assigned_schema>
```

Replace `<your_snowflake_account_identifier>` with the account the instructor gives you, `<your_snowflake_username>` and `<your_snowflake_password>` with your own Snowflake login, and `<your_assigned_schema>` with your personal schema (`TECHCATALYST.<your_name>`).

### What is `authenticator`, and the browser fallback

The connector needs to know **how** to prove who you are. By default, when a `password` is present, it uses Snowflake's username-and-password sign-in (internally the `snowflake` authenticator). That is what the config above does, and what we use in class.

There is another option you should know about: adding the line `authenticator = externalbrowser`. That switches to **browser-based single sign-on**. Instead of reading a password, the connector opens a web browser so you sign in through an identity provider (Okta, AD FS, or another SAML provider), and no password is stored anywhere.

Use the browser option as a **fallback if your password sign-in fails** (for example, your account is configured for SSO). To switch, edit `snow.cfg`: remove the `password` line and add `authenticator = externalbrowser`.

```ini
[DEV]
account = <your_snowflake_account_identifier>
user = <your_snowflake_username>
authenticator = externalbrowser
role = DE
warehouse = COMPUTE_WH
database = TECHCATALYST
schema = <your_assigned_schema>
```

Nothing else in this notebook changes. `**params` simply passes `authenticator` to `connect` instead of `password`. The loader below accepts either sign-in style.

The repository-root `.gitignore` already ignores every `snow.cfg`, so your password (or browser session) stays on your machine. Keep secrets in `snow.cfg` only, never in this notebook, and do not create a nested `.gitignore`.

In [ ]:
CONFIG_PATH = Path("snow.cfg")
CONFIG_SECTION = "DEV"

config = ConfigParser()
loaded_files = config.read(CONFIG_PATH)

assert loaded_files, (
    f"Could not read {CONFIG_PATH.resolve()}. "
    "Complete Activity 0 and keep snow.cfg beside this notebook."
)
assert CONFIG_SECTION in config, (
    f"Missing [{CONFIG_SECTION}] section in {CONFIG_PATH}."
)

params = {
    key: value.strip()
    for key, value in config[CONFIG_SECTION].items()
    if value.strip()
}

# These keys are always required, whichever sign-in style you use.
REQUIRED_CONNECTION_KEYS = {
    "account",
    "user",
    "role",
    "warehouse",
    "database",
    "schema",
}

missing_connection_keys = REQUIRED_CONNECTION_KEYS.difference(params)
assert not missing_connection_keys, (
    f"Missing connection keys: {sorted(missing_connection_keys)}"
)

# You must sign in with EITHER a password OR browser SSO.
uses_browser = params.get("authenticator") == "externalbrowser"
assert "password" in params or uses_browser, (
    "Add a password to snow.cfg, or set authenticator = externalbrowser "
    "to sign in with the browser."
)

placeholder_keys = {
    key
    for key, value in params.items()
    if value.startswith("<") or value.endswith(">")
}
assert not placeholder_keys, (
    f"Replace placeholders for: {sorted(placeholder_keys)}"
)

# Show the context WITHOUT the password, so nothing secret prints.
safe_context = {
    key: params[key]
    for key in ["user", "role", "warehouse", "database", "schema"]
    if key in params
}
if "authenticator" in params:
    safe_context["authenticator"] = params["authenticator"]
display(safe_context)

### Connect using dictionary unpacking

`**params` unpacks the dictionary so its keys become named arguments to `snowflake.connector.connect`.

For example, `{"account": "abc", "user": "sam"}` becomes `connect(account="abc", user="sam")`.

`connect` returns a **connection object**. You reuse it for every query in this notebook and for `write_pandas`, and you close it at the end with `conn.close()`. This connector is the core Snowflake Python SDK. Next week you will meet **Snowpark**, a higher-level Snowflake API that lets you work with DataFrames directly on Snowflake; it is built on top of the same connection ideas you are learning here.

Complete the one connection line, then run the context verification query.

In [ ]:
conn = snowflake.connector.connect(**params)
print("Snowflake connection opened.")

with conn.cursor() as cursor:
    cursor.execute(
        '''
        SELECT
          CURRENT_USER(),
          CURRENT_ROLE(),
          CURRENT_WAREHOUSE(),
          CURRENT_DATABASE(),
          CURRENT_SCHEMA()
        '''
    )
    connection_context = cursor.fetchone()

print({
    "user": connection_context[0],
    "role": connection_context[1],
    "warehouse": connection_context[2],
    "database": connection_context[3],
    "schema": connection_context[4],
})


## 9. Create explicit transient target tables

You already created tables in SQL earlier today. Apply that DDL skill here.

The first target statement is completed as a model. Write the other three statements so their columns and types match the DataFrames exactly.

Use:

- `VARCHAR` for text
- `BOOLEAN` for flags
- `NUMBER` for counts and quarter
- `FLOAT` for average severity

Do not let the connector guess the target schema.


In [ ]:
TARGET_DDL = {
    "SEVERITY_ANALYSIS": """
        CREATE OR REPLACE TRANSIENT TABLE SEVERITY_ANALYSIS (
          WEATHER_CONDITION VARCHAR,
          ROAD_CLASSIFICATION VARCHAR,
          ACCIDENT_COUNT NUMBER,
          AVERAGE_SEVERITY_SCORE FLOAT
        )
    """,
    "FATAL_TEMPORAL_PATTERNS": """
        CREATE OR REPLACE TRANSIENT TABLE FATAL_TEMPORAL_PATTERNS (
          TIME_OF_DAY_CATEGORY VARCHAR,
          WEEKEND_FLAG BOOLEAN,
          NUMBER_OF_FATAL_ACCIDENTS NUMBER
        )
    """,
    "CAUSE_VEHICLE_ANALYSIS": """
        CREATE OR REPLACE TRANSIENT TABLE CAUSE_VEHICLE_ANALYSIS (
          ACCIDENT_CAUSE VARCHAR,
          MULTI_VEHICLE_FLAG BOOLEAN,
          NUMBER_OF_ACCIDENTS NUMBER,
          AVERAGE_SEVERITY_SCORE FLOAT
        )
    """,
    "QUARTERLY_HOTSPOT_ANALYSIS": """
        CREATE OR REPLACE TRANSIENT TABLE QUARTERLY_HOTSPOT_ANALYSIS (
          STATE_CODE VARCHAR,
          QUARTER NUMBER,
          TOTAL_ACCIDENTS NUMBER
        )
    """,
}


In [ ]:
assert set(TARGET_DDL) == set(ANALYSES)

with conn.cursor() as cursor:
    for table_name, ddl_statement in TARGET_DDL.items():
        cursor.execute(ddl_statement)
        print(f"Created {table_name}")


## 10. Write each DataFrame with `write_pandas`

`write_pandas` is a helper from `snowflake.connector.pandas_tools`. Behind one Python call it writes your DataFrame to local Parquet files, uploads them to a temporary Snowflake stage with `PUT`, and loads the table with `COPY INTO`. Those are the same stage and `COPY INTO` ideas from Activity 2, now driven from Python.

It returns a tuple of four values: `(success, num_chunks, num_rows, output)`.

- `success`: did the load succeed
- `num_chunks`: how many chunks it copied
- `num_rows`: how many rows it wrote
- `output`: the raw `COPY INTO` result

In the loop:

1. Copy the DataFrame.
2. Uppercase its columns so they match the unquoted Snowflake DDL.
3. Call `write_pandas`.
4. Pass `quote_identifiers=False` because the explicit target columns are unquoted uppercase identifiers. (Its default is `True`, which would wrap every name in double quotes and no longer match your table.)
5. Use `on_error="ABORT_STATEMENT"` so a bad row stops that statement instead of silently loading partial data.
6. Store success, chunk count, DataFrame rows, and written rows.

In [ ]:
snowflake_outputs = {
    table_name: frame.copy()
    for table_name, (frame, _) in ANALYSES.items()
}

write_results = []

for table_name, frame in snowflake_outputs.items():
    snowflake_frame = frame.copy()
    snowflake_frame.columns = [
        column.upper()
        for column in snowflake_frame.columns
    ]

    success, chunk_count, written_rows, copy_output = write_pandas(
        conn=conn,
        df=snowflake_frame,
        table_name=table_name,
        quote_identifiers=False,
        on_error="ABORT_STATEMENT",
    )

    write_results.append({
        "table_name": table_name,
        "success": success,
        "chunk_count": chunk_count,
        "dataframe_rows": len(snowflake_frame),
        "written_rows": written_rows,
        "copy_output": copy_output,
    })

write_results_df = pd.DataFrame(write_results)
display(write_results_df.drop(columns="copy_output"))


## 11. Reconcile Snowflake row counts

For each table, compare:

- pandas DataFrame rows
- rows reported by `write_pandas`
- `SELECT COUNT(*)` from Snowflake

A pipeline is not complete until all three counts match.


In [ ]:
written_row_lookup = {
    record["table_name"]: record["written_rows"]
    for record in write_results
}
success_lookup = {
    record["table_name"]: record["success"]
    for record in write_results
}

reconciliation_records = []

with conn.cursor() as cursor:
    for table_name, frame in snowflake_outputs.items():
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        snowflake_rows = cursor.fetchone()[0]
        dataframe_rows = len(frame)
        written_rows = written_row_lookup[table_name]

        reconciliation_records.append({
            "table_name": table_name,
            "dataframe_rows": dataframe_rows,
            "written_rows": written_rows,
            "snowflake_rows": snowflake_rows,
            "counts_match": (
                dataframe_rows
                == written_rows
                == snowflake_rows
            ),
        })

reconciliation = pd.DataFrame(reconciliation_records)
display(reconciliation)

assert all(success_lookup.values())
assert reconciliation["counts_match"].all()


### Expected reconciliation

| Table | DataFrame rows | Written rows | Snowflake rows | Match |
|---|---:|---:|---:|---|
| SEVERITY_ANALYSIS | 28 | 28 | 28 | True |
| FATAL_TEMPORAL_PATTERNS | 8 | 8 | 8 | True |
| CAUSE_VEHICLE_ANALYSIS | 167 | 167 | 167 | True |
| QUARTERLY_HOTSPOT_ANALYSIS | 108 | 108 | 108 | True |


In [ ]:
conn.close()
print("Snowflake connection closed.")


## 12. Reflection

Add a Markdown cell and answer:

1. Which work happened in pandas, and which work happened in Snowflake?
2. Why did you build the rename dictionary instead of receiving a completed mapping?
3. Why was the IQR implementation provided, and what interpretation work remained yours?
4. Why did the pipeline pre-create transient tables instead of letting a library guess the schema?
5. What does `**params` do in the connection call?
6. How does `write_pandas` connect to the temporary stage and `COPY INTO` concepts from Activity 2?
7. Which analysis would you rebuild as a Snowflake view, and why?
